# StashFace — مطابقة على Kaggle (2x T4 GPU)

**قبل ما تشغل أي خلية:** من إعدادات الـnotebook (يمين الشاشة) اختار Accelerator = **GPU T4 x2**.

شغّل الخلايا بالترتيب من فوق لتحت.

In [ ]:
# ============================================================
# خلية الإعداد — عدّل القيم دي بس، والباقي متلمسوش
# ============================================================

# رابط الـGitHub repo بتاع المشروع
GITHUB_REPO_URL = "PUT_YOUR_GITHUB_REPO_URL_HERE"

# الـtoken بتاع حسابك على Hugging Face (لازم يكون عنده صلاحية Write)
HF_TOKEN = "PUT_YOUR_HF_TOKEN_HERE"

# الـdataset اللي فيه ملف الأداء، وهيتحفظ فيه ناتج المطابقة كمان في الآخر
HF_DATASET_ID = "abdelwahabnabil500/datafile"

# اسم ملف بيانات الأداء جوه الـdataset
HF_INPUT_FILENAME = "tpdb_all_performers.json"

# اسم ملف النتيجة اللي هيتولّد، وهيترفع بنفس الاسم ده تاني للـdataset في الآخر
HF_OUTPUT_FILENAME = "matched_identities.json"

In [ ]:
# ============================================================
# تحميل الكود وتثبيت المكتبات — مفيش حاجة تتعدل هنا
# ============================================================
import os

# 1) تحميل الكود من الـGitHub repo
!git clone {GITHUB_REPO_URL} /kaggle/working/stashface_pipeline
%cd /kaggle/working/stashface_pipeline

# 2) تثبيت المكتبات المطلوبة
!pip install -q -r requirements.txt

# 3) استبدال onnxruntime بنسخة الـGPU — عشان يستخدم الـT4 فعليًا مش الـCPU
!pip uninstall -y -q onnxruntime
!pip install -q onnxruntime-gpu

# 4) تسجيل الدخول لـHugging Face بالـtoken بتاعك
from huggingface_hub import login
login(token=HF_TOKEN)
os.environ["HF_TOKEN"] = HF_TOKEN

In [ ]:
# ============================================================
# تحميل بيانات المشروع (الموديل + قاعدة البيانات) + ملف الأداء
# ============================================================

# 5) تحميل بيانات المشروع (adaface model + performers.zvec) من الـbucket
!python setup.py --skip-install

# 6) تحميل ملف tpdb_all_performers.json من الـdataset بتاعك
from huggingface_hub import hf_hub_download
import shutil

local_path = hf_hub_download(
    repo_id=HF_DATASET_ID,
    repo_type="dataset",
    filename=HF_INPUT_FILENAME,
    token=HF_TOKEN,
)
shutil.copy(local_path, "tpdb_all_performers.json")
print("تم تحميل ملف البيانات:", local_path)

## تجربة سريعة (اختياري بس مستحسن)

شغّل الخلية دي الأول للتأكد إن كل حاجة شغالة صح قبل ما تشغل على الـ114 ألف صورة كلهم.

In [ ]:
!python run_kaggle.py --limit 50

## التشغيل الكامل على كل البيانات

دي هتاخد وقت طويل (ساعات، حسب حجم البيانات). لو الجلسة اتقفلت أو حصل أي كراش، رجّع شغّل نفس الخلية تاني — هيكمل من حيث ما وقف من غير ما يعيد اللي خلص منه.

In [ ]:
!python run_kaggle.py

## رفع ملف النتيجة (أهم خطوة)

شغّل الخلية دي بس بعد ما الخلية اللي فاتت تخلص تمامًا.

In [ ]:
from huggingface_hub import HfApi

assert os.path.exists("matched_identities.json"), "الملف مش موجود — تأكد إن خلية التشغيل الكامل خلصت من غير أخطاء"

api = HfApi(token=HF_TOKEN)
api.upload_file(
    path_or_fileobj="matched_identities.json",
    path_in_repo=HF_OUTPUT_FILENAME,
    repo_id=HF_DATASET_ID,
    repo_type="dataset",
    token=HF_TOKEN,
    commit_message="Add matched_identities.json from stashface pipeline run",
)
print("تم رفع ملف النتائج بنجاح إلى:", HF_DATASET_ID)